In [61]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
import spacy
import re
import contractions
from textblob import TextBlob
import os

In [62]:
df=PyPDFLoader("unit2.pdf").load()

In [63]:
data=""
for i in df:
    data+=i.page_content
data
    


"Unit 2\nData Exploration\nData Exploration is the initial step in data science where the dataset is analyzed, \nsummarized, and visualized to understand its structure, patterns, trends, relationships, \nand anomalies before applying any modeling techniques.\nIt helps the data scientist to ask the right questions and choose appropriate preprocessing \nand models.\nObjectives of Data Exploration\n1. Understand data structure and size\n2. Identify data types (numerical, categorical)\n3. Detect missing values and outliers\n4. Study distribution and spread of data\n5. Identify relationships between variables\n6. Validate assumptions for modeling\nType of exploration is always done based on the data you are using , We can classify data exploration \ninto two types:\n\uf0b7 Univariate Data Exploration.\n\uf0b7 Multivariate Data Exploration. \nUnivariate Data Exploration \nUnivariate analysis involves analyzing a single variable at a time to understand its \ndistribution, central tendency, an

## Text Normalization

In [64]:
data=data.lower()
type(data)

str

In [65]:
data=contractions.fix(data) # expanding the word

In [66]:
data=str(TextBlob(data).correct()) # correcting the words


In [67]:
data=re.sub(r'[^a-zA-Z0-9\s]'," ",data)
data

'unit 2\ndata exploration\ndata exploration is the initial step in data science where the dataset is analyzed  \nsummarized  and visualized to understand its structure  patterns  tends  relationships  \nand anomalies before applying any modeling technique \nit helps the data scientist to ask the right questions and choose appropriate preprocessing \nand models \nobjectives of data exploration\n1  understand data structure and size\n2  identify data types  numerical  categorical \n3  detect missing values and outlines\n4  study distribution and spread of data\n5  identify relationships between variable\n6  validity assumptions for modeling\ntype of exploration is always done based on the data you are using   we can classify data exploration \ninto two types \n  univariate data exploration \n  multivariate data exploration  \nunivariate data exploration \nunivariate analysis involves analyzing a single variable at a time to understand its \ndistribution  central tendency  and amiability 

In [68]:
data=re.sub(r'[\n]'," ",data)
data

'unit 2 data exploration data exploration is the initial step in data science where the dataset is analyzed   summarized  and visualized to understand its structure  patterns  tends  relationships   and anomalies before applying any modeling technique  it helps the data scientist to ask the right questions and choose appropriate preprocessing  and models  objectives of data exploration 1  understand data structure and size 2  identify data types  numerical  categorical  3  detect missing values and outlines 4  study distribution and spread of data 5  identify relationships between variable 6  validity assumptions for modeling type of exploration is always done based on the data you are using   we can classify data exploration  into two types    univariate data exploration    multivariate data exploration   univariate data exploration  univariate analysis involves analyzing a single variable at a time to understand its  distribution  central tendency  and amiability  technique used is  

In [69]:
data=re.sub(r'[\s]{1,}'," ",data)
data

'unit 2 data exploration data exploration is the initial step in data science where the dataset is analyzed summarized and visualized to understand its structure patterns tends relationships and anomalies before applying any modeling technique it helps the data scientist to ask the right questions and choose appropriate preprocessing and models objectives of data exploration 1 understand data structure and size 2 identify data types numerical categorical 3 detect missing values and outlines 4 study distribution and spread of data 5 identify relationships between variable 6 validity assumptions for modeling type of exploration is always done based on the data you are using we can classify data exploration into two types univariate data exploration multivariate data exploration univariate data exploration univariate analysis involves analyzing a single variable at a time to understand its distribution central tendency and amiability technique used is measures of central tendency measures

## Lemmatization 

removing the stop words

In [70]:
nlp=spacy.load('en_core_web_sm')
tokens=nlp(data)
tokens

unit 2 data exploration data exploration is the initial step in data science where the dataset is analyzed summarized and visualized to understand its structure patterns tends relationships and anomalies before applying any modeling technique it helps the data scientist to ask the right questions and choose appropriate preprocessing and models objectives of data exploration 1 understand data structure and size 2 identify data types numerical categorical 3 detect missing values and outlines 4 study distribution and spread of data 5 identify relationships between variable 6 validity assumptions for modeling type of exploration is always done based on the data you are using we can classify data exploration into two types univariate data exploration multivariate data exploration univariate data exploration univariate analysis involves analyzing a single variable at a time to understand its distribution central tendency and amiability technique used is measures of central tendency measures 

In [71]:
updated_tokens=[str(token.lemma_) for token in tokens if not token.is_stop]
data=" ".join(updated_tokens)
data

'unit 2 datum exploration datum exploration initial step data science dataset analyze summarize visualize understand structure pattern tend relationship anomaly apply modeling technique help data scientist ask right question choose appropriate preprocessing model objective datum exploration 1 understand datum structure size 2 identify data type numerical categorical 3 detect miss value outline 4 study distribution spread datum 5 identify relationship variable 6 validity assumption modeling type exploration base datum classify datum exploration type univariate data exploration multivariate data exploration univariate data exploration univariate analysis involve analyze single variable time understand distribution central tendency amiability technique measure central tendency measure spread measure central tendency measure central tendency statistical average tell point item tendency cluster measure central tendency know statistical average follow popular average mean know arithmetic ave

## Chunking

In [72]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=100,
    separators=['\n\n','.','\n',' ','']
)
chunks=splitter.create_documents([data])
chunks[0].metadata={'filename':'unit2.pdf'} # information about the file
chunks

[Document(metadata={'filename': 'unit2.pdf'}, page_content='unit 2 datum exploration datum exploration initial step data science dataset analyze summarize visualize understand structure pattern tend relationship anomaly apply modeling technique help data'),
 Document(metadata={}, page_content='visualize understand structure pattern tend relationship anomaly apply modeling technique help data scientist ask right question choose appropriate preprocessing model objective datum exploration 1'),
 Document(metadata={}, page_content='scientist ask right question choose appropriate preprocessing model objective datum exploration 1 understand datum structure size 2 identify data type numerical categorical 3 detect miss value'),
 Document(metadata={}, page_content='1 understand datum structure size 2 identify data type numerical categorical 3 detect miss value outline 4 study distribution spread datum 5 identify relationship variable 6 validity assumption'),
 Document(metadata={}, page_content='

## Chunk Embeddings (convert chunks to vecors)

In [73]:
embedding_model=HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-miniLM-L6-v2'
)
embedding_model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-miniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='sentence-transformers/all-miniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [74]:
vectordb=FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
vectordb

In [75]:
user_query="What are the types of correlations ?"
r_chunks=vectordb.similarity_search(user_query)
r_chunks

[Document(id='b2361934-c290-4b1e-9ab1-3cc514a87533', metadata={}, page_content='study tend achieve high score exam type correlation positive correlation variable direction e increase increase vice versa relation call positive correlation negative correlation variable opposite'),
 Document(id='bc141d8b-81de-4500-b0f9-d11b2b1959a1', metadata={}, page_content='measure relationship correlation repression measure relationship statistical measure relationship variable set datum example generally high relationship correlation parent s education academic'),
 Document(id='b8e0f1b1-2402-48f4-8c99-cb2d7e1633fa', metadata={}, page_content='sale increase example type datum suppose advertise want compare popularity advertisement webster correlation analysis term correlation combination word co relation quantity correlation observe change'),
 Document(id='e970a7ea-920f-4b52-87bc-295269139dad', metadata={}, page_content='increase vice versa relation call positive correlation negative correlation varia

In [96]:
retrieval_chunks=[chunks.page_content for chunks in r_chunks]
retrieval_text=" ".join(retrieval_chunks)
retrieval_text

'study tend achieve high score exam type correlation positive correlation variable direction e increase increase vice versa relation call positive correlation negative correlation variable opposite measure relationship correlation repression measure relationship statistical measure relationship variable set datum example generally high relationship correlation parent s education academic sale increase example type datum suppose advertise want compare popularity advertisement webster correlation analysis term correlation combination word co relation quantity correlation observe change increase vice versa relation call positive correlation negative correlation variable opposite direction e increase decrease vice versa relation call negative correlation constant change variable'

In [97]:
updated_r_chunks=set()
for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)

R_text =' '.join(updated_r_chunks)
R_text

'measure relationship correlation repression measure relationship statistical measure relationship variable set datum example generally high relationship correlation parent s education academic study tend achieve high score exam type correlation positive correlation variable direction e increase increase vice versa relation call positive correlation negative correlation variable opposite sale increase example type datum suppose advertise want compare popularity advertisement webster correlation analysis term correlation combination word co relation quantity correlation observe change increase vice versa relation call positive correlation negative correlation variable opposite direction e increase decrease vice versa relation call negative correlation constant change variable'

In [98]:
def retrieval_text(query:str,k=3)->str:
    r_chunks=vectordb.similarity_search(user_query,k=k)
    return " ".join([chunks.page_content for chunks in r_chunks])
try:
    r_text=retrieval_text("What are the types of correlations ?",3)
except Exception as e:
    print("Error: ",e)

In [99]:
model_ollama = ChatOllama(
    model='qwen2.5:1.5b',
    temperature=0
)
model_gemini  = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash',
    api_key = os.environ['GEMINI_API_KEY'],
    temperature=0
)

In [106]:
def generate_text(query,answer):
    prompt = f'''
    You're a helpful assistant and structure the given context.
    input : {query}
    output : {answer}
    
    Note:
    1) Don't add any extra contents just structure my given output.
    2) If the query is not related to context just say 'query is not related to context'.
    3) don't use heading or bullet points
    4) keep the answers short and meaningful
    5) Return exactly this format:  
    input : {query}
    output : structured output    
    '''
    try:
        response=model_gemini.invoke(prompt)
        return response.content[0]['text']
    except :
        response=model_ollama.invoke(prompt)
        return response.content
user_query="Explain data exploration"
user_query=re.sub(r'[^a-zA-Z0-9\s]'," ",user_query)
retrieval_t=retrieval_text(user_query,3)
final_response=generate_text(user_query,retrieval_t)
print(final_response)

input : Explain data exploration
output : Data exploration is the initial step in data science where a dataset is analyzed, summarized, and visualized to understand its structure, patterns, trends, relationships, and anomalies before applying modeling techniques. Data exploration is classified into univariate data exploration and multivariate data exploration. Univariate analysis involves analyzing a single variable at a time to understand its distribution and central tendency measurement techniques.
